# Trajectory and Tool Evals (Evaluating Intermediate Steps, Tool Selection, and Arguments)
Because agents operate in multi-step loops where they dynamically decide which tools to call and what arguments to pass, evaluating only the final output is like grading a math test by looking only at the final number while ignoring all intermediate work. If the student made a fundamental error on step two, getting the right answer at the end might have been pure luck.

In this topic, we dive deep into how to evaluate intermediate steps, tool selection accuracy, and argument validity.

## 1. The Anatomy of an Agent Trajectory
An agent trajectory (or trace) is a sequence of alternating reasoning and execution steps. A typical trace looks like this:


[User Prompt] ──► [Thought 1] ──► [Tool Call: get_weather(loc="Tokyo")]
                                            │
                                            ▼
                    [Observation: "22°C, Rainy"] ◄── (Environment)
                                            │
                                            ▼
                  [Thought 2] ──► [Tool Call: convert_temp(c=22)]
                                            │
                                            ▼
                 [Observation: "71.6°F"] ◄── (Environment)
                                            │
                                            ▼
                               [Final Response to User]


To evaluate this trajectory, we break it down into three distinct evaluation checkpoints:

Tool Selection Evaluation: Did the agent call the correct tools in a logical sequence?

Argument Validation Evaluation: Were the arguments passed to those tools syntactically and semantically valid?

Trajectory Efficiency Evaluation: Did the agent reach the goal efficiently, or did it get trapped in circular loops?

## 2. Evaluating Tool Selection Accuracy
When an agent is given a toolbox of 10 different tools, choosing the right tool for a specific sub-task is critical. We measure tool selection using standard classification metrics adapted for agents:

Exact Match (Tool Sequence): Did the agent call the exact sequence of tools defined in the golden test case? (e.g., ["search_db", "summarize"] vs ["search_db", "search_db", "summarize"]).

Tool Recall: Of all the mandatory tools required to solve the task, did the agent invoke all of them?

Hallucinated Tool Calls: Did the agent attempt to call a tool that does not exist in its registry? (A common LLM failure mode).

## 3. Evaluating Tool Arguments (Schema & Semantic Validity)
Even if the agent picks the correct tool, passing incorrect arguments will cause the tool execution to fail or return corrupted data.

Schema Compliance: Does the argument payload strictly match the tool's expected JSON schema (types, required fields, enum constraints)?

Semantic Argument Accuracy: Are the values passed inside the arguments correct based on the context? (e.g., if the user asked for Tokyo weather, did the agent pass location="Tokyo" or did it hallucinate location="Kyoto"?).

## 4. Hands-On Python Pattern: Evaluating Trajectories and Tool Calls
Below is a hands-on Python testing snippet that evaluates an agent's recorded trace against golden criteria for tool selection and argument correctness.

In [ ]:
import json
from typing import List, Dict, Any

# --- 1. Define Golden Test Case Criteria ---
GOLDEN_CRITERIA = {
    "expected_tools": ["query_database", "format_report"],
    "required_argument_checks": {
        "query_database": lambda args: "sql" in args and "SELECT" in args["sql"],
        "format_report": lambda args: args.get("format") == "markdown"
    }
}

# --- 2. Simulate an Agent's Executed Trajectory ---
agent_execution_trace = [
    {
        "step": 1,
        "tool_called": "query_database",
        "arguments": {"sql": "SELECT * FROM users WHERE active = true;"}
    },
    {
        "step": 2,
        "tool_called": "format_report",
        "arguments": {"format": "markdown", "include_header": True}
    }
]

# --- 3. Evaluation Harness ---
def evaluate_trajectory(trace: List[Dict[str, Any]], criteria: Dict[str, Any]) -> Dict[str, Any]:
    # Check 1: Tool Selection Sequence Match
    executed_tools = [step["tool_called"] for step in trace]
    expected_tools = criteria["expected_tools"]
    tool_sequence_match = (executed_tools == expected_tools)
    
    # Check 2: Argument Validation per Tool
    argument_validity = {}
    for step in trace:
        tool = step["tool_called"]
        args = step["arguments"]
        
        if tool in criteria["required_argument_checks"]:
            # Run the custom validation lambda function
            validator = criteria["required_argument_checks"][tool]
            argument_validity[tool] = validator(args)
        else:
            argument_validity[tool] = True # No special constraints

    all_args_valid = all(argument_validity.values())
    
    # Overall Trajectory Score
    overall_score = 1.0 if (tool_sequence_match and all_args_valid) else 0.0

    return {
        "score": overall_score,
        "tool_sequence_match": tool_sequence_match,
        "executed_tools": executed_tools,
        "argument_validity_details": argument_validity
    }

# --- Run Evaluation ---
if __name__ == "__main__":
    result = evaluate_trajectory(agent_execution_trace, GOLDEN_CRITERIA)
    print("--- Trajectory Evaluation Report ---")
    print(json.dumps(result, indent=2))

## 5. Key Takeaways
Trace Logging is Mandatory: Emphasize to students that writing evaluation scripts for trajectories is impossible unless their agent framework records intermediate tool calls and arguments into structured logs.

Catch Errors Early: Point out that evaluating tool selection and arguments allows developers to isolate failures immediately—knowing whether an agent failed because it picked the wrong tool versus passing a bad argument.